In [1]:
# ruff: noqa: F401, F403

import os
import subprocess
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

from IPython.display import *

from pacer import (
    CoordinateSystem,
    # read_dat_file,
    DatVersion,
    GPMFSource,
    GPSSample,
    Lap,
    Laps,
    Point,
    PointInTime_GPSSample,
    RawGPSSource,
    Segment,
    SequentialGPSSource,
    Vec3f,
)

In [6]:
file_paths = [
    "/Volumes/Untitled/DCIM/100GOPRO/GX010111.MP4",
]
single_files = [GPMFSource(f) for f in file_paths]

samples = []
start_of_file = {}
end_of_file = {}


for fn, f in zip(file_paths, single_files):
    total_duration = f.get_total_duration()

    def on_sample(s, _, _2):
        if fn not in start_of_file:
            start_of_file[fn] = pd.Timestamp(s.timestamp_ms, unit="ms")
        end_of_file[fn] = pd.Timestamp(s.timestamp_ms, unit="ms")
        if s.full_speed > 3:
            samples.append(s)

    while not f.is_end():
        f.read_samples(on_sample)
        f.next()

In [ ]:
import re
import xml.etree.ElementTree as ET


tcx_path = "/Users/denys/Downloads/2025-11-08-frol-mk-dmax-frol.tcx"

samples = []


with open(tcx_path, "r") as xml_file:
    xml_str = xml_file.read()
    xml_str = re.sub(' xmlns="[^"]+"', "", xml_str, count=1)
    root = ET.fromstring(xml_str)
    activities = root.findall(".//Activity")
    for activity in activities:
        print("-- {} --".format(activity.attrib["Sport"]))
        prev_distance = 0.0
        tracking_points = activity.findall(".//Trackpoint")
        for tracking_point in list(tracking_points):
            distance = float(tracking_point.find("./DistanceMeters").text)
            sample = GPSSample(
                altitude=float(tracking_point.find("./AltitudeMeters").text),
                lon=float(tracking_point.find("./Position/LongitudeDegrees").text),
                lat=float(tracking_point.find("./Position/LatitudeDegrees").text),
                full_speed=distance - prev_distance,
                timestamp_ms=pd.to_datetime(tracking_point.find("./Time").text).value
                // 10**6,
            )
            prev_distance = distance

            print(
                {
                    "time": pd.to_datetime(tracking_point.find("./Time").text),
                    "hr": int(tracking_point.find("./HeartRateBpm/Value").text),
                    "sample": sample,
                }
            )
            samples.append(sample)

-- Biking --
{'time': Timestamp('2025-11-08 12:51:29+0000', tz='UTC'), 'hr': 97, 'sample': GPSSample(lat=52.040365, lon=-0.784948, altitude=77.599998, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=1762606289000)}
{'time': Timestamp('2025-11-08 12:51:30+0000', tz='UTC'), 'hr': 98, 'sample': GPSSample(lat=52.040365, lon=-0.784948, altitude=77.599998, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=1762606290000)}
{'time': Timestamp('2025-11-08 12:51:41+0000', tz='UTC'), 'hr': 101, 'sample': GPSSample(lat=52.040365, lon=-0.784948, altitude=77.599998, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=1762606301000)}
{'time': Timestamp('2025-11-08 12:51:45+0000', tz='UTC'), 'hr': 102, 'sample': GPSSample(lat=52.040365, lon=-0.784948, altitude=77.599998, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=1762606305000)}
{'time': Timestamp('2025-11-08 12:52:00+0000', tz='UTC'), 'hr': 102, 'sample': GPSSample(lat=52.040365, lon=-0.784948, altitude=77.599998, fu

In [4]:
tracking_point.find("./Position/LatitudeDegrees").text

'52.040394991636276'

In [14]:
cs = CoordinateSystem(samples[0])

laps = Laps()
laps.set_coordinate_system(cs)

start_time = pd.to_datetime(samples[0].timestamp_ms, unit="ms")
for s in samples:
    t = (pd.to_datetime(s.timestamp_ms, unit="ms") - start_time).total_seconds()
    laps.add_point(s, t)


def locate_timestamp(timestamp: pd.Timestamp | float):
    if not isinstance(timestamp, pd.Timestamp):
        timestamp = start_time + pd.to_timedelta(timestamp, unit="s")
    assert isinstance(timestamp, pd.Timestamp)
    for f in file_paths:
        if start_of_file[f] <= timestamp < end_of_file[f]:
            return f, timestamp - start_of_file[f]


fig = px.line(x=[cs.local(s).x for s in samples], y=[cs.local(s).y for s in samples])
fig.add_trace(
    px.line(
        x=[laps.sectors.start_line.first.x, laps.sectors.start_line.second.x],
        y=[laps.sectors.start_line.first.y, laps.sectors.start_line.second.y],
    ).data[0]
)

# Play around to set right start line.
s1, s2 = Point(x=-15, y=55), Point(x=0, y=30)
# s3, s4 = Point(x=100, y=-80), Point(x=105, y=-65)
# s5, s6 = Point(x=191, y=-160), Point(x=197, y=-190)

fig.add_trace(px.line(x=[s1.x, s2.x], y=[s1.y, s2.y]).data[0])
# fig.add_trace(px.line(x=[s3.x, s4.x], y=[s3.y, s4.y]).data[0])
# fig.add_trace(px.line(x=[s5.x, s6.x], y=[s5.y, s6.y]).data[0])
fig.update_layout(height=800)

In [8]:
type(laps.sectors.sector_lines)

list

In [15]:
laps.sectors.start_line = Segment(s1, s2)
laps.sectors.sector_lines = [
    # Segment(s3, s4),
    # Segment(s5, s6),
]
laps.update()

laps_times = pd.DataFrame(
    [
        dict(
            lap=i,
            lap_time=laps.lap_time(i),
            **{f"S{j + 1}": laps.sector_time(3 * i + j) for j in range(3)},
        )
        for i in range(laps.laps_count())
    ]
)

# Filter out laps around pits
non_outliars = laps_times["lap_time"] > 10

# Filter out slow laps: in-between sessions, bunch of yellows, etc.
non_outliars &= (
    laps_times["lap_time"] < 1.25 * laps_times.loc[non_outliars, "lap_time"].min()
)

px.line(
    laps_times.where(non_outliars),
    x="lap",
    y="lap_time",
    title=r"Lap times (whitin 125% of the best)",
    markers=True,
)

In [57]:
lap10 = laps.get_lap(10)
lap15 = laps.get_lap(15)

lap10.width = 5
lap15.width = 5

lap15_10 = lap10.resample(lap15, cs)

In [58]:
len(lap10.points), len(lap15.points), len(lap15_10.points)

(762, 764, 762)

In [59]:
px.line(pd.DataFrame({
    "dist": lap10.cum_distances,
    "lap10": [p.point.full_speed for p in lap10.points],
    "lap12": [p.point.full_speed for p in lap15_10.points],
}).set_index("dist"))

In [60]:
t10 = pd.Series([p.time for p in lap10.points])
t11 = pd.Series([p.time for p in lap15_10.points])

px.line(x=lap10.cum_distances, y=(t10 - t11).pipe(lambda s: s - s.iloc[0]))

In [62]:
data = pd.DataFrame(
    dict(
        x=np.array(cs.local(p.point).x for p in lap10.points),
        y=np.array(cs.local(p.point).y for p in lap10.points),
        x2=np.array(cs.local(p.point).x for p in lap15_10.points),
        y2=np.array(cs.local(p.point).y for p in lap15_10.points),
        cum_dist=lap10.cum_distances,
        lap10_time=np.array([p.time - lap10.points[0].time for p in lap10.points]),
        lap15_time=np.array([p.time - lap15_10.points[0].time for p in lap15_10.points]),
        lap10_speed=np.array([p.point.full_speed for p in lap10.points]),
        lap15_speed=np.array([p.point.full_speed for p in lap15_10.points]),
    )
).assign(
    delta=lambda d: d["lap10_time"]
    - d["lap15_time"]
    - d["lap10_time"].min()
    + d["lap15_time"].min(),
    lap10_time=lambda d: d["lap10_time"],
    lap15_time=lambda d: d["lap15_time"],
)

In [63]:
px.line(data.set_index("cum_dist"), y="delta", title="Lap 10.time - Lap 15.time")

In [64]:
fig = px.scatter(
    data.assign(ddelta=lambda d: d["delta"].diff().rolling(20).mean().shift(-10)),
    x="x",
    y="y",
    color="ddelta",
    hover_data=["cum_dist", "lap10_speed", "lap15_speed"],
    title="(smoothed) Derivative of delta in space. Positive = lap1 is losing time",
).update_layout(height=500, margin=dict(l=0, r=0, t=30, b=0))
for trace in (
    px.scatter(
        data.assign(ddelta=lambda d: -d["delta"].diff().rolling(20).mean().shift(-10)),
        x="x2",
        y="y2",
        color="ddelta",
        hover_data=["cum_dist", "delta", "lap10_speed", "lap15_speed"],
        title="(smoothed) Derivative of delta in space. Positive = lap1 is losing time",
    )
    .update_layout(height=500, margin=dict(l=0, r=0, t=30, b=0))
    .data
):
    fig.add_trace(trace)
fig.update_layout(height=600)

In [44]:
df = pd.DataFrame({
    "lat1": [p.point.lat for p in lap10.points], "lon1": [p.point.lon for p in lap10.points], "dist": [p for p in lap10.cum_distances]
})
px.scatter(df, x="lon1", y="lat1", color="dist", title="Lap 10")

In [12]:
px.histogram(good_sectors["S1"], nbins=30, title="Sector 1 times distribution")

In [13]:
px.histogram(good_sectors["S2"], nbins=14, title="Sector 2 times distribution")

In [14]:
px.histogram(good_sectors["S3"], nbins=30, title="Sector 3 times distribution")

In [15]:
raw_laptimes = """01 1:27.798 [18]
02 1:29.447 [17]
03 1:30.355 [17]
04 1:33.718 [17]
05 1:32.349 [16]
06 1:14.794 [16]
07 1:21.772 [16]
08 1:16.177 [16]
09 1:17.225 [15]
10 1:12.886 [15]
11 1:12.401 [15]
12 1:16.090 [14]
13 1:23.844 [14]
14 1:15.042 [15]
15 1:12.077 [15]
16 1:13.518 [15]
17 1:13.378 [14]
18 1:17.909 [15]
19 1:13.696 [14]
20 1:17.433 [15]
21 1:24.206 [15]
22 1:13.007 [15]
23 1:14.224 [16]
24 1:10.666 [16]
25 1:10.995 [15]
26 1:12.946 [15]
27 1:10.541 [15]
28 1:09.988 [14]
29 1:12.646 [14]
30 1:10.985 [14]
31 1:16.647 [14]
"""

official_laptimes = pd.DataFrame(
    [
        {
            "lap": int(line.split(" ")[0]),
            "lap_time": pd.to_timedelta("0:" + line.split(" ")[1]).total_seconds(),
            "position": int(re.search(r"\[(\d+)\]", line).group(1)),
        }
        for line in raw_laptimes.strip().split("\n")
    ]
)
comparison = official_laptimes.assign(
    guessed_laptime=laps_times.iloc[:-1]
    .iloc[-(len(official_laptimes)) :]["lap_time"]
    .values
)

In [16]:
comparison.assign(diff_seconds=comparison["lap_time"] - comparison["guessed_laptime"])[
    ["lap", "position", "lap_time", "guessed_laptime", "diff_seconds"]
]

,lap,position,lap_time,guessed_laptime,diff_seconds
0,1,18,87.798,87.575387,0.222613
1,2,17,89.447,89.867382,-0.420382
2,3,17,90.355,89.997406,0.357594
3,4,17,93.718,94.006442,-0.288442
4,5,16,92.349,92.006400,0.342600
5,6,16,74.794,74.906053,-0.112053
6,7,16,81.772,81.634098,0.137902
7,8,16,76.177,76.361253,-0.184253
8,9,15,77.225,76.983396,0.241604
9,10,15,72.886,72.819496,0.066504


In [17]:
px.histogram(
    comparison["lap_time"] - comparison["guessed_laptime"],
    title="Difference between official and guessed lap times (seconds)",
)

In [18]:
laps_times.iloc[:-1].iloc[-(len(official_laptimes)) :]

,lap,lap_time,S1,S2,S3
5,5,87.575387,20.336977,31.814026,35.424385
6,6,89.867382,19.712480,33.589494,36.565409
7,7,89.997406,22.425588,30.669669,36.902150
8,8,94.006442,18.017138,33.650071,42.339234
9,9,92.006400,23.607023,34.743276,33.656100
10,10,74.906053,16.924699,26.750185,31.231169
11,11,81.634098,16.462314,28.954798,36.216986
12,12,76.361253,18.599654,26.575135,31.186463
13,13,76.983396,20.173796,26.366182,30.443418
14,14,72.819496,16.267761,26.261207,30.290529


In [19]:
lap1 = laps.get_lap(32)
lap2 = laps.get_lap(34)
lap1.points[0].time, lap2.points[0].time
locate_timestamp(lap1.points[0].time), locate_timestamp(lap1.points[-1].time)
locate_timestamp(lap1.points[0].time), locate_timestamp(lap2.points[0].time)
lap1.width = 10
lap21 = lap1.resample(lap2, cs)
start_time + pd.Timedelta(seconds=lap1.points[0].time)

Timestamp('2025-11-08 13:50:24.872534381')

In [20]:
lap1.count(), lap21.count()

(60, 60)

In [24]:
data = pd.DataFrame(
    dict(
        x=np.array(cs.local(p.point).x for p in lap1.points),
        y=np.array(cs.local(p.point).y for p in lap1.points),
        cum_dist=lap1.cum_distances,
        lap1_time=np.array([p.time for p in lap1.points]),
        lap2_time=np.array([p.time for p in lap21.points]),
        lap1_speed=np.array([p.point.full_speed for p in lap1.points]),
        lap2_speed=np.array([p.point.full_speed for p in lap21.points]),
    )
).assign(
    delta=lambda d: d["lap1_time"]
    - d["lap2_time"]
    - d["lap1_time"].min()
    + d["lap2_time"].min(),
    lap1_time=lambda d: d["lap1_time"],
    lap2_time=lambda d: d["lap2_time"],
)


lap1_time = data["lap1_time"].max() - data["lap1_time"].min()
lap2_time = data["lap2_time"].max() - data["lap2_time"].min()

px.line(
    data,
    x="cum_dist",
    y="delta",
    hover_data=["lap1_time", "lap2_time"],
    title=f"lap1_time ({lap1_time:.2f}) - lap2_time ({lap2_time:.2f})",
)


In [23]:
data

,x,y,cum_dist,lap1_time,lap2_time,lap1_speed,lap2_speed,delta
0,11.418309,-10.982559,0.000000,3535.872534,3678.562177,36.921012,33.995718,0.000000
1,9.883121,-12.950196,2.495709,3536.000000,3678.593799,38.429688,34.367034,0.095843
2,8.545892,-33.069262,22.660155,3537.000000,3679.689507,19.410156,24.341956,0.000136
3,15.702919,-51.817135,42.736620,3538.000000,3680.693091,19.750000,19.398879,-0.003449
4,42.67249,-83.304563,84.197150,3540.000000,3682.747168,40.269531,37.352013,-0.057525
5,57.790162,-89.650642,100.592781,3541.000000,3683.685318,16.910156,23.756296,0.004325
6,71.271996,-86.053978,114.547553,3542.000000,3684.681005,13.949219,14.737745,0.008637
7,84.392235,-82.634985,128.111824,3543.000000,3685.694226,13.050781,13.045213,-0.004584
8,95.308704,-88.720089,140.609734,3544.000000,3687.021052,12.230469,8.944201,-0.331410
9,103.814701,-97.499049,152.835194,3545.000000,3687.930337,10.429688,12.439266,-0.240695


In [22]:
px.line(
    data,
    x="cum_dist",
    y=["lap1_speed", "lap2_speed"],
    title=f"lap1_time ({lap1_time:.2f}), lap2_time ({lap2_time:.2f})",
)


In [ ]:
locate_timestamp(800), locate_timestamp(1896)

In [95]:
px.scatter(
    data.assign(ddelta=lambda d: d["delta"].diff().rolling(1).mean()),
    x="x",
    y="y",
    color="ddelta",
    hover_data=["cum_dist", "lap1_speed", "lap2_speed"],
).update_layout(height=600)

In [51]:
(
    locate_timestamp(start_time + pd.Timedelta("591s")),
    locate_timestamp(start_time + pd.Timedelta("1947s")),
)

(('/Users/denys/Downloads/GX010108.MP4', Timedelta('0 days 00:10:13.900000')),
 ('/Users/denys/Downloads/GX030108.MP4', Timedelta('0 days 00:07:12.399000')))

In [ ]:
px.scatter(
    pd.DataFrame(
        dict(
            x=[cs.local(p.point).x for p in lap1.points],
            y=[cs.local(p.point).y for p in lap1.points],
            full_speed=[p.point.full_speed for p in lap1.points],
            cum_distance=lap1.cum_distances,
        )
    ),
    x="x",
    y="y",
    hover_data="cum_distance",
    color="delta",
).update_layout(height=600)

In [92]:
lap1.lap_time(), lap21.lap_time()

(48.32459112347226, 48.29960160144947)

In [23]:
fig = px.line(x=[cs.local(s).x for s in samples], y=[cs.local(s).y for s in samples])
fig.add_trace(
    px.line(
        x=[laps.sectors.start_line.first.x, laps.sectors.start_line.second.x],
        y=[laps.sectors.start_line.first.y, laps.sectors.start_line.second.y],
    ).data[0]
)
fig.add_trace(px.line(x=[s1.x, s2.x], y=[s1.y, s2.y]).data[0])

In [42]:
dir(laps.sectors)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'sector_lines',
 'start_line']

In [47]:
fig = px.line(x=[s.lat for s in samples], y=[s.lon for s in samples])

In [4]:
df = pd.DataFrame(
    [
        dict(
            lat=s.lat,
            lon=s.lon,
            alt=s.altitude,
            full_speed=s.full_speed,
            ground_speed=s.ground_speed,
            timestamp=pd.to_datetime(s.timestamp_ms, unit="ms"),
            begin=b,
            end=e,
        )
        for s, b, e in samples
    ]
)

In [130]:
df

,lat,lon,alt,full_speed,ground_speed,timestamp,begin,end
0,51.375703,-0.361656,22.978,0.07,0.056,2025-09-06 14:03:40.399,0.000,1.001
1,51.375705,-0.361657,23.221,0.06,0.073,2025-09-06 14:03:40.499,0.000,1.001
2,51.375706,-0.361658,23.402,0.08,0.070,2025-09-06 14:03:40.599,0.000,1.001
3,51.375707,-0.361659,23.567,0.09,0.050,2025-09-06 14:03:40.699,0.000,1.001
4,51.375709,-0.361660,23.766,0.06,0.054,2025-09-06 14:03:40.799,0.000,1.001
...,...,...,...,...,...,...,...,...
7682,51.376140,-0.361179,15.849,20.11,19.595,2025-09-06 14:16:28.599,767.767,768.768
7683,51.376129,-0.361199,15.887,19.60,19.230,2025-09-06 14:16:28.699,767.767,768.768
7684,51.376117,-0.361219,15.925,19.23,18.865,2025-09-06 14:16:28.799,767.767,768.768
7685,51.376105,-0.361236,15.963,18.87,18.022,2025-09-06 14:16:28.899,767.767,768.768


In [129]:
px.line((df["timestamp"] - df["timestamp"].min()).dt.total_seconds() - df["begin"])